# Clase 06 — Estimación e intervalos de confianza

**Curso:** Estadística Aplicada con Python · **Bloque 3:** Inferencia
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/06_estimacion_intervalos_confianza.md`

## Qué vas a ver en esta clase

- **Estimador** y **estimación**: la media de una muestra como aproximación a la media de la población.
- La **distribución muestral** y el **error estándar** de la media (σ/√n, y s/√n cuando σ es desconocida).
- El **intervalo de confianza (IC)** para la media con la t de Student, con un cálculo a mano.
- La **interpretación correcta** del IC, comprobada con un experimento de **cobertura**.
- Cómo cambia el ancho del IC con el nivel de confianza y con el tamaño de la muestra.
- El **bootstrap percentil** para la media y la mediana, y sus **límites**.
- Un apunte sobre el IC de una **proporción** (Wilson).

## Requisitos

- Clase 1 (variabilidad de muestreo y sesgo) y clase 5 (t de Student, teorema del límite central).
- Librerías: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; ya vienen instaladas en Google Colab).

> **Cómo usar este notebook:** ejecuta las celdas en orden. Antes de cada experimento escribe qué esperas que ocurra. Los tiempos de entrega son **sintéticos** (*ejemplo ilustrativo*): no son datos reales de ninguna empresa.

---
## Celda 0 — Setup

Importamos las librerías y fijamos la **semilla** del generador aleatorio: con la misma semilla, todos obtenemos los mismos números.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)          # semilla fija: todos obtenemos los mismos resultados
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
print("Librerías cargadas correctamente.")

---
## 1. Una población conocida para poder comprobar todo

Para *ver* qué hace un estimador necesitamos conocer la verdad. Generamos una **población sintética** de 100 000 pedidos con su tiempo de entrega en horas: la distribución es **asimétrica** (muchas entregas rápidas y una cola de entregas lentas), como suele ocurrir con tiempos de espera. En la vida real nunca conocemos la población; aquí sí, y por eso podemos comprobar si nuestros intervalos aciertan.

In [ ]:
poblacion = rng.lognormal(mean=np.log(30), sigma=0.5, size=100_000)     # horas (ejemplo ilustrativo)
MU = poblacion.mean()          # parámetro: la media verdadera
SIGMA = poblacion.std()        # parámetro: la desviación típica verdadera
MEDIANA_POB = np.median(poblacion)

print(f"Media verdadera μ          = {MU:.2f} h")
print(f"Mediana verdadera          = {MEDIANA_POB:.2f} h")
print(f"Desviación típica verdadera σ = {SIGMA:.2f} h")
print(f"Asimetría (skew)           = {stats.skew(poblacion):.2f}")

plt.hist(poblacion, bins=80, range=(0, 150), color="steelblue", alpha=0.8)
plt.axvline(MU, color="crimson", linestyle="--", label="media")
plt.axvline(MEDIANA_POB, color="black", linestyle=":", label="mediana")
plt.title("Población sintética de tiempos de entrega (ejemplo ilustrativo)")
plt.xlabel("Horas"); plt.ylabel("Pedidos"); plt.legend()
plt.show()

**Observa:** la media (≈ 34 h) queda a la derecha de la mediana (≈ 30 h) por la cola larga. Recuerda de la clase 1 que una muestra es solo una parte de la población: la media de una muestra será un número parecido a μ, pero no idéntico.

---
## 2. Estimador, estimación y distribución muestral

El **estimador** es la regla (aquí, "calcula la media de la muestra"); la **estimación** es el número que sale de *una* muestra concreta. Tomamos una muestra de 50 pedidos y, después, repetimos el muestreo 2 000 veces para ver cómo varía la estimación: eso es la **distribución muestral** de la media (clase 1, ahora la cuantificamos).

In [ ]:
N = 50
una_muestra = rng.choice(poblacion, size=N)
print(f"Estimación con UNA muestra: media = {una_muestra.mean():.2f} h   (μ = {MU:.2f} h)")

medias = rng.choice(poblacion, size=(2_000, N)).mean(axis=1)       # 2 000 muestras de 50
print(f"Media de las 2 000 medias        : {medias.mean():.2f} h")
print(f"Desviación típica de las 2 000 medias: {medias.std(ddof=1):.2f} h   ← error estándar empírico")
print(f"σ/√n = {SIGMA / np.sqrt(N):.2f} h                       ← error estándar teórico")

plt.hist(medias, bins=40, color="darkorange", alpha=0.85)
plt.axvline(MU, color="crimson", linestyle="--", label="μ")
plt.title("Distribución muestral de la media (n = 50, 2 000 muestras)")
plt.xlabel("Media de la muestra (h)"); plt.ylabel("Frecuencia"); plt.legend()
plt.show()

**¿Qué observas?**

- ¿Se centra la distribución de las medias en μ? ¿Qué nombre tiene esa propiedad (clase 1)?
- La población es muy asimétrica; ¿y la distribución de las medias? Relaciónalo con el teorema del límite central (clase 5).
- El error estándar empírico y el teórico σ/√n, ¿coinciden?

*(Tu respuesta aquí.)*

### El error estándar y el tamaño de la muestra

Repetimos el experimento con distintos n. En la práctica solo tenemos **una** muestra, así que σ se estima con la desviación típica muestral s (con `ddof=1`) y el error estándar estimado es s/√n.

In [ ]:
filas = []
for n in [10, 40, 160, 640]:
    muestras = rng.choice(poblacion, size=(2_000, n))
    filas.append({"n": n,
                  "error estándar empírico": muestras.mean(axis=1).std(ddof=1),
                  "teórico σ/√n": SIGMA / np.sqrt(n),
                  "estimado con 1 muestra s/√n": muestras[0].std(ddof=1) / np.sqrt(n)})
display(pd.DataFrame(filas).set_index("n").round(2))

**Observa:** cada vez que multiplicas n por 4, el error estándar se divide por 2 (no por 4). Para reducir el error a la mitad hace falta cuadruplicar los datos: los rendimientos son decrecientes.

---
## 3. El intervalo de confianza para la media: un cálculo a mano

Un IC del 95 % para la media es **media ± t · s/√n**, donde t es el cuantil 0,975 de la t de Student con n − 1 grados de libertad. Lo calculamos paso a paso con 8 tiempos de entrega (*ejemplo ilustrativo*), de modo que puedas repetirlo con lápiz y papel.

In [ ]:
pequena = np.array([24, 31, 28, 45, 27, 52, 30, 36])          # 8 tiempos de entrega, en horas
n = len(pequena)
media = pequena.mean()
s = pequena.std(ddof=1)
ee = s / np.sqrt(n)
t_crit = stats.t.ppf(0.975, df=n - 1)
semiancho = t_crit * ee

print(f"n = {n}   media = {media:.3f}   s = {s:.3f}   s/√n = {ee:.3f}")
print(f"t(0,975; {n - 1} g.l.) = {t_crit:.3f}   semiancho = {semiancho:.3f}")
print(f"IC 95 % a mano        : ({media - semiancho:.2f}, {media + semiancho:.2f})")

ic_scipy = stats.t.interval(0.95, df=n - 1, loc=media, scale=ee)
print(f"IC 95 % con scipy     : ({ic_scipy[0]:.2f}, {ic_scipy[1]:.2f})")

z_crit = stats.norm.ppf(0.975)
print(f"IC con z (INCORRECTO con n = 8): ({media - z_crit * ee:.2f}, {media + z_crit * ee:.2f})   ← demasiado estrecho")

**Observa:** la fórmula a mano y `scipy` dan lo mismo. Con solo 8 datos, t = 2,365 es bastante mayor que el 1,96 de la normal: la t "paga" la incertidumbre añadida por haber estimado σ con s. Usar z aquí produciría un intervalo demasiado estrecho.

---
## 4. Experimento guiado: ¿qué significa "95 % de confianza"?

Tomamos **1 000 muestras** de 50 pedidos, construimos **1 000 IC al 95 %** y contamos cuántos contienen la media verdadera μ. Como aquí conocemos μ, podemos comprobarlo. Antes de ejecutar, escribe qué proporción esperas.

In [ ]:
N_REP, N_MUESTRA, NIVEL = 1_000, 50, 0.95

muestras = rng.choice(poblacion, size=(N_REP, N_MUESTRA))
medias = muestras.mean(axis=1)
ees = muestras.std(axis=1, ddof=1) / np.sqrt(N_MUESTRA)
t_crit = stats.t.ppf(1 - (1 - NIVEL) / 2, df=N_MUESTRA - 1)
inferior, superior = medias - t_crit * ees, medias + t_crit * ees

contiene = (inferior <= MU) & (MU <= superior)
cobertura = contiene.mean()
ee_cob = np.sqrt(NIVEL * (1 - NIVEL) / N_REP)

print(f"Intervalos que contienen μ: {contiene.sum()} de {N_REP}")
print(f"Cobertura observada            : {cobertura:.3f}   (nominal {NIVEL}; error estándar de la cobertura ≈ {ee_cob:.3f})")

# los 50 primeros intervalos, con los que fallan resaltados
fig, ax = plt.subplots(figsize=(8, 6))
for i in range(50):
    color = "seagreen" if contiene[i] else "crimson"
    ax.hlines(i, inferior[i], superior[i], color=color, linewidth=2)
    ax.plot(medias[i], i, "o", color=color, markersize=3)
ax.axvline(MU, color="black", linestyle="--", label="μ (verdadera)")
ax.set_title("50 intervalos de confianza al 95 %: en rojo, los que NO contienen μ")
ax.set_xlabel("Horas"); ax.set_ylabel("Muestra número"); ax.legend()
plt.show()

**¿Qué observas?**

- ¿La cobertura observada está cerca de 0,95? Con datos asimétricos y n = 50 suele quedar algo por debajo: la t asume una distribución de la media casi normal, y con esta cola larga es solo aproximada.
- En el gráfico, ¿cuántos de los 50 intervalos fallan? ¿Se pueden distinguir de los demás *sin conocer μ*?
- Para **un** intervalo concreto ya calculado, μ está dentro o no lo está: no hay "95 % de probabilidad" en ese caso. ¿A qué se refiere entonces el 95 %?

*(Tu respuesta aquí.)*

---
## 5. Qué hace más ancho o más estrecho un intervalo

Dos palancas: el **nivel de confianza** (a más confianza, intervalo más ancho) y el **tamaño de la muestra** (a más datos, más estrecho). Lo vemos con una muestra fija y luego promediando sobre muchas.

In [ ]:
muestra = poblacion[:50]                                           # una muestra fija
ee_m = muestra.std(ddof=1) / np.sqrt(len(muestra))
filas = []
for nivel in [0.90, 0.95, 0.99]:
    lo, hi = stats.t.interval(nivel, df=len(muestra) - 1, loc=muestra.mean(), scale=ee_m)
    filas.append({"nivel": nivel, "límite inferior": lo, "límite superior": hi, "ancho": hi - lo})
display(pd.DataFrame(filas).set_index("nivel").round(2))

anchos = []
for n in [10, 20, 40, 80, 160, 320]:
    m = rng.choice(poblacion, size=(500, n))
    semi = stats.t.ppf(0.975, n - 1) * m.std(axis=1, ddof=1) / np.sqrt(n)
    anchos.append(2 * semi.mean())
plt.plot([10, 20, 40, 80, 160, 320], anchos, "o-", label="ancho medio del IC 95 %")
plt.xscale("log"); plt.yscale("log")
plt.title("El ancho del IC baja como 1/√n")
plt.xlabel("n"); plt.ylabel("Ancho (h)"); plt.legend()
plt.show()

**Observa:** el IC al 99 % es más ancho que el del 95 %, que a su vez es más ancho que el del 90 %. Un intervalo más estrecho **no es "mejor"**: con los mismos datos, para estrecharlo solo hay que aceptar menos confianza. La única forma de estrechar *sin* perder confianza es conseguir más datos.

---
## 6. Bootstrap percentil

Para la media hay una fórmula; para la **mediana** no hay una fórmula sencilla. El *bootstrap* usa la propia muestra como sustituto de la población: **remuestrea con reposición** n datos de la muestra, calcula el estadístico, y repite 10 000 veces. Los percentiles 2,5 y 97,5 de esos valores forman el IC. Lo vectorizamos: una matriz de índices de tamaño (B, n) evita bucles.

In [ ]:
def bootstrap_dist(datos, estadistico, B, rng):
    """Distribución bootstrap: B remuestreos con reposición."""
    idx = rng.integers(0, len(datos), size=(B, len(datos)))
    return estadistico(datos[idx], axis=1)

def ic_percentil(dist, nivel=0.95):
    """IC percentil: cuantiles alfa/2 y 1 - alfa/2 de la distribución bootstrap."""
    alfa = 1 - nivel
    return np.quantile(dist, [alfa / 2, 1 - alfa / 2])

muestra = rng.choice(poblacion, size=50)
B = 10_000
dist_media = bootstrap_dist(muestra, np.mean, B, rng)
dist_mediana = bootstrap_dist(muestra, np.median, B, rng)

ic_boot_media = ic_percentil(dist_media)
ic_boot_mediana = ic_percentil(dist_mediana)
ee_m = muestra.std(ddof=1) / np.sqrt(len(muestra))
ic_t = stats.t.interval(0.95, df=len(muestra) - 1, loc=muestra.mean(), scale=ee_m)
media_m, mediana_m = muestra.mean(), np.median(muestra)

print(f"Media de la muestra   = {media_m:.2f}   IC t = ({ic_t[0]:.2f}, {ic_t[1]:.2f})   IC bootstrap = ({ic_boot_media[0]:.2f}, {ic_boot_media[1]:.2f})")
print(f"Mediana de la muestra = {mediana_m:.2f}   IC bootstrap = ({ic_boot_mediana[0]:.2f}, {ic_boot_mediana[1]:.2f})")
print(f"Verdad en la población: μ = {MU:.2f}, mediana = {MEDIANA_POB:.2f}")

fig, ejes = plt.subplots(1, 2, figsize=(10, 4))
for ax, dist, ic, titulo, real in [(ejes[0], dist_media, ic_boot_media, "Bootstrap de la media", MU),
                                   (ejes[1], dist_mediana, ic_boot_mediana, "Bootstrap de la mediana", MEDIANA_POB)]:
    ax.hist(dist, bins=50, color="steelblue", alpha=0.85)
    ax.axvline(ic[0], color="black", linestyle="--"); ax.axvline(ic[1], color="black", linestyle="--")
    ax.axvline(real, color="crimson", label="valor verdadero")
    ax.set_title(titulo); ax.set_xlabel("Horas"); ax.legend()
plt.tight_layout(); plt.show()

**¿Qué observas?**

- El IC bootstrap de la media, ¿se parece al de la t? ¿Por qué es razonable que se parezcan?
- La distribución bootstrap de la mediana, ¿es más "dentada" o irregular que la de la media? (Pista: la mediana de n datos solo puede tomar valores que ya están en la muestra.)
- ¿Contienen los dos intervalos el valor verdadero de la población en esta muestra? Recuerda que eso, en una sola muestra, es cuestión de suerte al 95 %.

*(Tu respuesta aquí.)*

---
## 7. Bootstrap frente a t: cobertura y límites

Comprobamos por simulación la cobertura de la **t** y del **bootstrap percentil** para la media, con la población asimétrica, en tres tamaños de muestra. En cada repetición: una muestra nueva, un IC t y un IC bootstrap (1 000 remuestreos). Usamos 500 repeticiones, así que el error estándar de cada cobertura es de aproximadamente 0,01.

In [ ]:
def cobertura_media(n, reps, B, rng):
    """Cobertura del IC t y del IC bootstrap percentil para la media."""
    aciertos_t = aciertos_b = 0
    t_crit = stats.t.ppf(0.975, n - 1)
    for _ in range(reps):
        m = rng.choice(poblacion, size=n)
        semi = t_crit * m.std(ddof=1) / np.sqrt(n)
        aciertos_t += (m.mean() - semi <= MU <= m.mean() + semi)
        lo, hi = ic_percentil(bootstrap_dist(m, np.mean, B, rng))
        aciertos_b += (lo <= MU <= hi)
    return aciertos_t / reps, aciertos_b / reps

filas = []
for n in [10, 30, 100]:
    cob_t, cob_b = cobertura_media(n, 500, 1_000, rng)
    filas.append({"n": n, "cobertura IC t": cob_t, "cobertura IC bootstrap": cob_b})
display(pd.DataFrame(filas).set_index("n").round(3))

**¿Qué observas?**

- Con n = 10, ninguno de los dos métodos llega al 95 %. ¿Cuál se queda más corto? El bootstrap no "arregla" una muestra muy pequeña: solo conoce los 10 datos que tiene.
- ¿Qué pasa al crecer n?
- Conclusión: con datos asimétricos y pocos datos, un IC "al 95 %" puede cubrir bastante menos; hay que decirlo y, si es posible, ampliar la muestra.

*(Tu respuesta aquí.)*

### El bootstrap no corrige una muestra sesgada

Simulamos un registro que **solo guarda las entregas de menos de 40 horas** (por ejemplo, porque las demás se gestionan en otro sistema). Es un sesgo de selección (clase 1): el bootstrap dará un intervalo muy "seguro de sí mismo" en el sitio equivocado.

In [ ]:
registro = poblacion[poblacion < 40]
muestra_sesgada = rng.choice(registro, size=200)
ic_sesgado = ic_percentil(bootstrap_dist(muestra_sesgada, np.mean, 10_000, rng))

print(f"Media de la muestra sesgada: {muestra_sesgada.mean():.2f} h")
print(f"IC bootstrap 95 %              : ({ic_sesgado[0]:.2f}, {ic_sesgado[1]:.2f})")
print(f"Media verdadera de la población: {MU:.2f} h   →   ¿dentro del IC? {ic_sesgado[0] <= MU <= ic_sesgado[1]}")

**Observa:** el intervalo es estrecho (200 datos) pero está lejos de μ. Ni la t ni el bootstrap corrigen un sesgo de selección: un IC mide solo la incertidumbre **de muestreo**, no la que introduce un mal diseño de recogida de datos.

---
## 8. (Opcional) IC para una proporción: Wilson

Para una proporción (por ejemplo, la fracción de entregas que tardan más de 48 h) el intervalo más recomendable es el de **Wilson**. Lo necesitaremos en las clases 7 y 13. `statsmodels` lo calcula con `proportion_confint`; lo comparamos con la fórmula normal simple (Wald).

In [ ]:
from statsmodels.stats.proportion import proportion_confint

p_real = (poblacion > 48).mean()
muestra_p = rng.choice(poblacion, size=200)
x_tardias = int((muestra_p > 48).sum())
print(f"Proporción verdadera de entregas > 48 h: {p_real:.3f}")
print(f"Muestra: {x_tardias} de 200 → proporción {x_tardias / 200:.3f}")

for metodo in ["wilson", "normal"]:
    lo, hi = proportion_confint(x_tardias, 200, alpha=0.05, method=metodo)
    print(f"  {metodo:7s} → ({lo:.3f}, {hi:.3f})")

print()
print("Caso extremo: 1 entrega tardía en 20")
p_hat = 1 / 20
lim_inf_wald = p_hat - 1.96 * np.sqrt(p_hat * (1 - p_hat) / 20)
print(f"  fórmula normal a mano → límite inferior = {lim_inf_wald:.3f}   ← ¡negativo!")
print(f"  Wilson → {tuple(round(v, 3) for v in proportion_confint(1, 20, alpha=0.05, method='wilson'))}")

**Observa:** con una proporción pequeña y pocos datos, la fórmula normal simple llega a dar un límite inferior negativo, imposible para una proporción. Wilson no tiene ese problema y tiene mejor cobertura.

---
## 9. Resumen de la clase

- Un **estimador** es una regla; una **estimación**, el número de una muestra concreta. Su variabilidad se resume en el **error estándar**: σ/√n, estimado con s/√n.
- El **IC para la media** con la t es media ± t · s/√n. Se usa t (no z) porque σ se estima con s, sobre todo con n pequeño.
- El **95 % describe el procedimiento**: en muestras repetidas, ≈ 95 % de los intervalos contiene μ. No es la probabilidad de que μ esté en *este* intervalo.
- Más confianza → intervalo más ancho; más datos → más estrecho (como 1/√n). Estrecho no significa mejor.
- El **bootstrap percentil** da IC para estadísticos sin fórmula sencilla (como la mediana), pero no crea información: falla con muestras muy pequeñas y no corrige sesgos.
- Para una **proporción**, Wilson (`proportion_confint`) es preferible a la fórmula normal simple.
- Un IC **no es un test**: comparar dos grupos con intervalos que se solapan o no es un atajo peligroso (clase 7).

**Siguiente paso:** el notebook de práctica de esta clase y, después, la clase 7 (tests de hipótesis I).